In [3]:
import os
import re
import sys
import pickle

sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from src.evaluation.metrics import tune_tfidf_params, tune_hybrid_weights
from src.recommender.hybrid import SVDHybridRecommender
from src.utils.preprocessing import (
    load_clean_data,
    preprocess_content_text,
    get_users_for_eval
)
from src.utils.mappings import get_toeic_part_mapping, get_subject_categories

np.random.seed(42)
sns.set_theme(style="whitegrid", palette="viridis")

In [4]:
# Load data
lectures_df, merged_df = load_clean_data()
print(f"Lectures: {len(lectures_df)}, Interactions: {len(merged_df)}")

Loading from paths:
Lectures: c:\Users\karat\Downloads\Data-Driven-Personalized-Educational-Content-Recommendation-System\data\cleaned\cleaned_lectures.csv
Merged: c:\Users\karat\Downloads\Data-Driven-Personalized-Educational-Content-Recommendation-System\data\cleaned\merged_cleaned_data.csv
Successfully loaded 1021 lectures and 117167 interactions
Lectures: 1021, Interactions: 117167


In [5]:
# Prepare bundle features (from notebooks/02_tf_idf.ipynb)
bundle_info = merged_df.groupby('bundle_id').agg({
    'part': 'first',
    'tags': lambda x: ';'.join(set(str(i) for i in x if pd.notna(i))),
    'question_id': lambda x: len(set(x))
}).reset_index()
bundle_info.columns = ['bundle_id', 'part', 'tags', 'question_count']
part_names = get_toeic_part_mapping()
bundle_info['part_name'] = bundle_info['part'].map(part_names)
bundle_info['subject_category'] = bundle_info['tags'].apply(lambda x: get_subject_categories(x)[0] if get_subject_categories(x) else "General")
bundle_info['content_text'] = (
    bundle_info['part_name'].fillna('') + ' ' +
    bundle_info['subject_category'].fillna('') + ' ' +
    bundle_info['tags'].fillna('')
)
bundle_info['content_text'] = bundle_info['content_text'].apply(preprocess_content_text)
bundle_info['content_text'].head()

0    part 1: listening - photographs general 1 2 17...
1    part 1: listening - photographs grammar_basic ...
2    part 1: listening - photographs grammar_basic ...
3    part 2: listening - question-response grammar_...
4    part 2: listening - question-response grammar_...
Name: content_text, dtype: object